<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.5-rag-with-langchain/notebooks/exercises-6_5.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.5: RAG with LangChain

*Module 6 · Retrieval-Augmented Generation (RAG)*

> In 6.1 we built RAG from scratch to understand every component. Now we rebuild it with LangChain — the most popular LLM framework — in a fraction of the code. Document loaders, text splitters, embeddings, vector stores, and retrieval chains — all plug-and-play with Gemini as the LLM backbone.

`LangChain` · `Gemini` · `ChromaDB` · `LCEL Chains` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-6.5.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — LangChain RAG in 15 Lines — The Express Version — `01_langchain_rag.py`
2. Step 2 — Document Loaders — LangChain Has 160+ Loaders — `02_loaders.py`
3. Step 3 — LCEL Chains — The Pipe Operator for LLM Apps — `03_lcel_chains.py`
4. Step 4 — Production RAG — Conversational Memory + Sources — `04_conversational_rag.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q langchain chromadb


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · LangChain RAG in 15 Lines — The Express Version

Document → split → embed → store → query. All LangChain components.

**`01_langchain_rag.py`** · _Block 1 of 4_

LANGCHAIN RAG — Full pipeline in 15 lines — pip install langchain langchain-google-genai langchain-chroma chromadb


In [ ]:
# LANGCHAIN RAG — Full pipeline in 15 lines
# pip install langchain langchain-google-genai langchain-chroma chromadb
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "your-key")

# 1. Documents
docs_text = [
    "Netsetos refund policy: full refund within 7 days. 50% from 8-30 days. No refunds after 30 days.",
    "GenAI course: 14999 rupees. Lifetime access, Discord, weekly live sessions, certificate, 5000 GCP credits.",
    "Live classes daily 7 PM IST on YouTube. Recordings in 2 hours. Python + GCP. 85% completion rate.",
    "Prerequisites: basic Python and high school math. No ML experience needed. 146 hours, 14 modules.",
]

# 2. Split
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = []
for text in docs_text:
    chunks.extend(splitter.create_documents([text]))

# 3. Embed + Store in ChromaDB
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 4. LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# 5. RAG Chain (LCEL)
prompt = ChatPromptTemplate.from_template(
    """Answer based ONLY on this context. If not found, say so.

Context: {context}

Question: {question}"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Query!
print("LangChain RAG:\n")
for q in ["What is the refund policy?", "How much does the course cost?", "What are the prerequisites?"]:
    answer = chain.invoke(q)
    print(f"  Q: {q}\n  A: {answer[:120]}\n")


> **What just happened?** The same 5-step RAG pipeline from 6.1 — but with LangChain components: RecursiveCharacterTextSplitter (chunking), GoogleGenerativeAIEmbeddings (embedding), Chroma (vector store), and an LCEL chain (retriever | format | prompt | LLM | parse). 15 lines replaced 60 lines. Same result. Same quality. But faster to write, easier to maintain, and you can swap any component.


### Step 2 · Document Loaders — LangChain Has 160+ Loaders

PDF, HTML, CSV, Google Drive, Notion, Confluence — one import each.

**`02_loaders.py`** · _Block 2 of 4_

LANGCHAIN DOCUMENT LOADERS — 160+ formats — pip install langchain-community pymupdf unstructured


In [ ]:
# LANGCHAIN DOCUMENT LOADERS — 160+ formats
# pip install langchain-community pymupdf unstructured

# ── Text files ──
from langchain_community.document_loaders import TextLoader
# loader = TextLoader("faq.txt")
# docs = loader.load()  # [{page_content: "...", metadata: {source: "faq.txt"}}]

# ── PDF files ──
from langchain_community.document_loaders import PyMuPDFLoader
# loader = PyMuPDFLoader("report.pdf")
# docs = loader.load()  # one Document per page, with page number metadata

# ── Web pages ──
from langchain_community.document_loaders import WebBaseLoader
# loader = WebBaseLoader("https://netsetos.com/faq")
# docs = loader.load()  # HTML → clean text automatically

# ── CSV files ──
from langchain_community.document_loaders.csv_loader import CSVLoader
# loader = CSVLoader("students.csv")
# docs = loader.load()  # one Document per row

# ── Directory of files ──
from langchain_community.document_loaders import DirectoryLoader
# loader = DirectoryLoader("./docs/", glob="**/*.txt")
# docs = loader.load()  # loads ALL .txt files in directory

print("LangChain Loaders:\n")
loaders = {
    "TextLoader": ".txt files",
    "PyMuPDFLoader": "PDF with page metadata",
    "WebBaseLoader": "Any URL → clean text",
    "CSVLoader": "CSV → one doc per row",
    "DirectoryLoader": "Entire folder of files",
    "NotionDBLoader": "Notion pages",
    "GoogleDriveLoader": "Google Drive docs",
    "UnstructuredHTMLLoader": "Complex HTML",
}
for name, desc in loaders.items():
    print(f"  {name:25s} → {desc}")
print(f"\n  Total: 160+ loaders. One import. One .load() call.")


### Step 3 · LCEL Chains — The Pipe Operator for LLM Apps

LangChain Expression Language: compose components with | like Unix pipes.

**`03_lcel_chains.py`** · _Block 3 of 4_

LCEL — LangChain Expression Language — Compose chains with the | pipe operator


In [ ]:
# LCEL — LangChain Expression Language
# Compose chains with the | pipe operator
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
import os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "your-key")
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# ── Chain 1: Simple prompt → LLM → string ──
simple = ChatPromptTemplate.from_template("Explain {topic} in one sentence.") | llm | StrOutputParser()
print("Chain 1 (simple):")
print(f"  {simple.invoke({'topic': 'RAG'})}\n")

# ── Chain 2: Multi-step (summarize then translate) ──
summarize = ChatPromptTemplate.from_template("Summarize in 20 words: {text}") | llm | StrOutputParser()
translate = ChatPromptTemplate.from_template("Translate to Hindi: {text}") | llm | StrOutputParser()

multi = {"text": summarize} | translate
print("Chain 2 (summarize → translate):")
print(f"  {multi.invoke({'text': 'RAG retrieves documents and injects them into LLM prompts for grounded answers.'})[:100]}\n")

# ── Chain 3: RAG chain with source tracking ──
rag_prompt = ChatPromptTemplate.from_template(
    "Context: {context}\n\nQuestion: {question}\n\nAnswer from context only:")

def fake_retriever(q):
    db = {"refund": "Full refund within 7 days.", "price": "GenAI course: 14999 rupees."}
    return next((v for k,v in db.items() if k in q.lower()), "No info found.")

rag_chain = (
    {"context": RunnableLambda(fake_retriever), "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print("Chain 3 (RAG):")
print(f"  {rag_chain.invoke('What is the refund policy?')}\n")

print("LCEL pattern: input | transform | prompt | llm | parser")
print("Every | is a composable step. Swap any component.")


> **What just happened?** Three chains, all built with the | pipe: (1) Simple: prompt → LLM → string. (2) Multi-step: summarize → translate (chain of chains). (3) RAG: retriever → format → prompt → LLM → parse. LCEL is LangChain’s composability model: every component is a Runnable. Connect them with pipes. Swap any piece. Add logging, streaming, or fallbacks by wrapping.


### Step 4 · Production RAG — Conversational Memory + Sources

Real apps need chat history and source citations. LangChain handles both.

**`04_conversational_rag.py`** · _Block 4 of 4_

CONVERSATIONAL RAG WITH SOURCES


In [ ]:
# CONVERSATIONAL RAG WITH SOURCES
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
import os

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "your-key")

# Build vectorstore
texts = [
    "Netsetos refund: full within 7 days, 50% 8-30 days, none after 30.",
    "GenAI course: 14999 rupees, 146 hours, 14 modules, Python + GCP.",
    "Live classes 7 PM IST daily. Recordings in 2 hours. 85% completion.",
    "Prerequisites: basic Python, high school math. No ML needed.",
]
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = splitter.create_documents(texts, metadatas=[{"source":f"doc_{i}"} for i in range(len(texts))])
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vs = Chroma.from_documents(docs, embeddings)
retriever = vs.as_retriever(search_kwargs={"k":2})

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# Conversational prompt with history
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer from context ONLY. If not found, say so.\n\nContext: {context}"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

# Simulate a conversation
history = []
print("Conversational RAG:\n")

for q in ["What are the prerequisites?", "How much does it cost?", "Can I get a refund if I don't like it?"]:
    # Retrieve
    retrieved = retriever.invoke(q)
    context = "\n".join(d.page_content for d in retrieved)
    sources = [d.metadata.get("source","?") for d in retrieved]

    # Generate with history
    messages = prompt.format_messages(context=context, history=history, question=q)
    response = llm.invoke(messages)
    answer = response.content.strip()

    # Update history
    history.append(HumanMessage(content=q))
    history.append(AIMessage(content=answer))

    print(f"  Q: {q}")
    print(f"  A: {answer[:120]}")
    print(f"  Sources: {sources}\n")


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
